<a href="https://colab.research.google.com/github/Metal-mouse10/qmbtools/blob/main/energy_pumping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np
from scipy.linalg import expm
from scipy.sparse import diags, eye, kron
import matplotlib.pyplot as plt
import os
import pandas as pd


# ============================================================
# PARAMETERS
# ============================================================
m_values = [-2.0, -1.8, 0.2, 0.5, 0.7, 1.8, 2.0]

lambda_values = [
    0.0,
    0.1,
    0.2,
    0.3,
    0.7,
    1.0,
    1.2,
    np.pi/2,
    1.7708,
    np.pi
]



dt = 0.05
tmax = 500

N_sat = 5


eta = 2.0

omega1 = 0.1
omega2 = omega1 * (1 + np.sqrt(5)) / 2

phi1 = np.pi / 10
phi2 = 0.0

times = np.arange(0, tmax, dt)

sin1 = np.sin(omega1*times + phi1)
cos1 = np.cos(omega1*times + phi1)

sin2 = np.sin(omega2*times + phi2)
cos2 = np.cos(omega2*times + phi2)

# ============================================================
# PAULI MATRICES
# ============================================================

sx = np.array([[0, 1],
               [1, 0]], dtype=complex)

sy = np.array([[0, -1j],
               [1j, 0]], dtype=complex)

sz = np.array([[1, 0],
               [0, -1]], dtype=complex)

# ============================================================
# HAMILTONIAN PIECES
# ============================================================

def create_Hd(lam, N_sat):

    m_vals = np.arange(N_sat + 1)

    Sz_eigs = (2 * m_vals - N_sat) / 2

    Sz_env = diags(Sz_eigs, 0)

    Sz_c = 0.5 * sz

    return (-lam * kron(Sz_env, Sz_c)).toarray()



def make_op(op):

    return kron(
        eye(N_sat + 1),
        0.5 * op
    ).toarray()

Sx_c = make_op(sx)
Sy_c = make_op(sy)
Sz_c = make_op(sz)

# ============================================================
# INITIAL STATE
# ============================================================

def initial_state(m):

    Hc0 = eta * (
        np.sin(phi1)*0.5*sx
        + np.sin(phi2)*0.5*sy
        + (m - np.cos(phi1) - np.cos(phi2))*0.5*sz
    )

    eigvals, eigvecs = np.linalg.eigh(Hc0)

    gs_central = eigvecs[:, np.argmin(eigvals)]

    env = np.zeros(N_sat+1, dtype=complex)
    env[N_sat] = 1.0

    return np.kron(env, gs_central)




def run_simulation(m, lam, psi0):

    Hd = create_Hd(lam, N_sat)

    psi = psi0.copy()

    mag_t = np.zeros(len(times))
    E1_t = np.zeros(len(times))
    E2_t = np.zeros(len(times))
    entropy_t = np.zeros(len(times))

    dH1 = eta * (
        omega1 * cos1[0] * Sx_c +
        omega1 * sin1[0] * Sz_c
    )

    dH2 = eta * (
        omega2 * cos2[0] * Sy_c +
        omega2 * sin2[0] * Sz_c
    )

    w1_old = np.real(np.vdot(psi, dH1 @ psi))
    w2_old = np.real(np.vdot(psi, dH2 @ psi))

    W1 = 0.0
    W2 = 0.0

    for i in range(len(times)):

        H_t = Hd + eta * (
            sin1[i] * Sx_c +
            sin2[i] * Sy_c +
            (m - cos1[i] - cos2[i]) * Sz_c
        )

        U = expm(-1j * H_t * dt)

        psi = U @ psi
        psi /= np.linalg.norm(psi)

        # Magnetization

        mag_t[i] = 2 * np.real(
            np.vdot(psi, Sz_c @ psi)
        )

  # =====================================================
# Entropy from reduced density matrix
# =====================================================

        psi_mat = psi.reshape(N_sat + 1, 2)

        rho_c = psi_mat.conj().T @ psi_mat

        eigvals = np.linalg.eigvalsh(rho_c)

        eigvals = eigvals[eigvals > 1e-14]

        entropy_t[i] = -np.sum(
            eigvals * np.log2(eigvals)
        )

        # Work

        dH1 = eta * (
            omega1 * cos1[i] * Sx_c +
            omega1 * sin1[i] * Sz_c
        )

        dH2 = eta * (
            omega2 * cos2[i] * Sy_c +
            omega2 * sin2[i] * Sz_c
        )

        w1_new = np.real(
            np.vdot(psi, dH1 @ psi)
        )

        w2_new = np.real(
            np.vdot(psi, dH2 @ psi)
        )

        W1 += 0.5 * (w1_old + w1_new) * dt
        W2 += 0.5 * (w2_old + w2_new) * dt

        E1_t[i] = W1
        E2_t[i] = W2

        w1_old = w1_new
        w2_old = w2_new

    return pd.DataFrame({
        "time": times,
        "mag": mag_t,
        "E1": E1_t,
        "E2": E2_t,
        "entropy": entropy_t
    })

# ============================================================
# MAIN SWEEP
# ============================================================

RESULT_DIR = "results"

os.makedirs(RESULT_DIR, exist_ok=True)

for m in m_values:

    print(f"\n===== m = {m} =====")

    # λ=0 ground state for this m
    psi0 = initial_state(m)

    m_folder = os.path.join(
        RESULT_DIR,
        f"m_{m}"
    )

    os.makedirs(m_folder, exist_ok=True)

    for lam in lambda_values:

        print(f"   λ = {lam:.4f}")

        df = run_simulation(
            m,
            lam,
            psi0
        )

        filename = os.path.join(
            m_folder,
            f"lambda_{lam:.4f}.csv"
        )

        df.to_csv(
            filename,
            index=False
        )

print("\nAll simulations completed.")


===== m = -2.0 =====
   λ = 0.0000
   λ = 0.1000
   λ = 0.2000
   λ = 0.3000
   λ = 0.7000
   λ = 1.0000
   λ = 1.2000
   λ = 1.5708
   λ = 1.7708
   λ = 3.1416

===== m = -1.8 =====
   λ = 0.0000
   λ = 0.1000
   λ = 0.2000
   λ = 0.3000
   λ = 0.7000
   λ = 1.0000
   λ = 1.2000
   λ = 1.5708
   λ = 1.7708
   λ = 3.1416

===== m = 0.2 =====
   λ = 0.0000
   λ = 0.1000
   λ = 0.2000
   λ = 0.3000
   λ = 0.7000
   λ = 1.0000
   λ = 1.2000
   λ = 1.5708
   λ = 1.7708
   λ = 3.1416

===== m = 0.5 =====
   λ = 0.0000
   λ = 0.1000
   λ = 0.2000
   λ = 0.3000
   λ = 0.7000
   λ = 1.0000
   λ = 1.2000
   λ = 1.5708
   λ = 1.7708
   λ = 3.1416

===== m = 0.7 =====
   λ = 0.0000
   λ = 0.1000
   λ = 0.2000
   λ = 0.3000
   λ = 0.7000
   λ = 1.0000
   λ = 1.2000
   λ = 1.5708
   λ = 1.7708
   λ = 3.1416

===== m = 1.8 =====
   λ = 0.0000
   λ = 0.1000
   λ = 0.2000
   λ = 0.3000
   λ = 0.7000
   λ = 1.0000
   λ = 1.2000
   λ = 1.5708
   λ = 1.7708
   λ = 3.1416

===== m = 2.0 =====
   λ = 0.00

In [15]:
import os
import pandas as pd
import matplotlib.pyplot as plt

RESULT_DIR = "results"
PDF_DIR = os.path.join(RESULT_DIR, "final_plots")

os.makedirs(PDF_DIR, exist_ok=True)

for m_folder in sorted(os.listdir(RESULT_DIR)):

    if not m_folder.startswith("m_"):
        continue

    m_path = os.path.join(RESULT_DIR, m_folder)

    if not os.path.isdir(m_path):
        continue

    m_value = m_folder.replace("m_", "")

    for csv_file in sorted(os.listdir(m_path)):

        if not csv_file.endswith(".csv"):
            continue

        df = pd.read_csv(
            os.path.join(m_path, csv_file)
        )

        lambda_str = csv_file.replace(".csv", "")

        fig, axs = plt.subplots(
            4, 1,
            figsize=(8, 10),
            sharex=True
        )

        axs[0].plot(df["time"], df["mag"])
        axs[0].set_ylabel("Mag")

        axs[1].plot(df["time"], df["E1"])
        axs[1].set_ylabel("E1")

        axs[2].plot(df["time"], df["E2"])
        axs[2].set_ylabel("E2")

        axs[3].plot(df["time"], df["entropy"])
        axs[3].set_ylabel("Entropy")
        axs[3].set_xlabel("Time")

        fig.suptitle(
            rf"$m={m_value}$, {lambda_str}"
        )

        plt.tight_layout()

        pdf_name = (
            f"m_{m_value}_{lambda_str}.pdf"
        )

        plt.savefig(
            os.path.join(PDF_DIR, pdf_name),
            bbox_inches="tight"
        )

        plt.close()

print("Finished creating all PDFs.")

Finished creating all PDFs.


In [17]:
import shutil

shutil.make_archive(
    "final_plots",
    "zip",
    "results/final_plots"
)

'/content/final_plots.zip'

In [18]:
from google.colab import files

files.download("final_plots.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>